In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [8]:
import json
import pandas as pd
from pathlib import Path

# =========================
# PATHS
# =========================

GOLD_JSON_PATH = "/content/drive/MyDrive/NLP_PRACTICA/golden_dataset_corregido.json"

RUNS_DIR = Path("/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a")
OUTPUT_DIR = Path("/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a_fixed")

# Crear carpeta destino si no existe
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# LOAD GOLD
# =========================

with open(GOLD_JSON_PATH, "r", encoding="utf-8") as f:
    gold_data = json.load(f)

if isinstance(gold_data, dict):
    for key, value in gold_data.items():
        if isinstance(value, list):
            gold_data = value
            break

gold_df = pd.DataFrame(gold_data)

gold_df = gold_df.rename(columns={
    "id": "idx",
    "pregunta": "question",
    "respuesta_breve": "gold_answer",
    "capitulos": "gold_chapters"
})

gold_df["idx"] = gold_df["idx"].astype(int)

gold_df["gold_chapters"] = gold_df["gold_chapters"].apply(
    lambda x: x if isinstance(x, list) else [x]
)

gold_map = dict(zip(gold_df["idx"], gold_df["gold_chapters"]))

# =========================
# FIND JSONL FILES
# =========================

jsonl_files = sorted([
    p for p in RUNS_DIR.glob("*.jsonl")
    if not p.name.endswith("_fixed.jsonl")
])

# =========================
# UPDATE JSONL FILES
# =========================

all_changes = []

for file_path in jsonl_files:
    print("\n" + "=" * 90)
    print(f"Procesando: {file_path.name}")

    df = pd.read_json(file_path, lines=True)

    if "idx" not in df.columns or "gold_chapters" not in df.columns:
        print("ERROR: columnas necesarias no encontradas")
        continue

    changes = []

    def normalize(x):
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        return [x]

    def update_row(row):
        idx = int(row["idx"])
        old = normalize(row["gold_chapters"])

        if idx not in gold_map:
            return old

        new = gold_map[idx]

        if sorted(old) != sorted(new):
            changes.append({
                "file": file_path.name,
                "idx": idx,
                "question": row.get("question", ""),
                "old": old,
                "new": new
            })
            all_changes.append(changes[-1])

        return new

    df["gold_chapters"] = df.apply(update_row, axis=1)

    print(f"Cambios: {len(changes)}")

    for c in changes[:10]:
        print(f"\nIDX {c['idx']}")
        print(f"OLD: {c['old']}")
        print(f"NEW: {c['new']}")

    # =========================
    # SAVE EN NUEVA CARPETA
    # =========================

    output_path = OUTPUT_DIR / (file_path.stem + "_fixed.jsonl")

    df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print(f"\nGuardado en: {output_path}")

# =========================
# RESUMEN GLOBAL
# =========================

changes_df = pd.DataFrame(all_changes)

print("\n" + "=" * 90)
print("TOTAL CAMBIOS:", len(changes_df))

if not changes_df.empty:
    display(changes_df)

    summary_path = OUTPUT_DIR / "changes_summary.csv"
    changes_df.to_csv(summary_path, index=False)

    print("\nResumen guardado en:")
    print(summary_path)
else:
    print("No hubo cambios.")


Procesando: gemma3_4b_4bit.jsonl
Cambios: 8

IDX 20
OLD: ['Arya I']
NEW: ['Jon II']

IDX 21
OLD: ['Arya I']
NEW: ['Jon II']

IDX 43
OLD: ['Tyrion III']
NEW: ['Tyrion III, Jon III']

IDX 62
OLD: ['Catelyn VII']
NEW: ['Catelyn VII, Tyrion V']

IDX 63
OLD: ['Catelyn VII']
NEW: ['Catelyn VII, Tyrion V']

IDX 72
OLD: ['Daenerys IV']
NEW: ['Daenerys V']

IDX 73
OLD: ['Daenerys IV']
NEW: ['Daenerys V']

IDX 91
OLD: ['Bran VI']
NEW: ['Catelyn X']

Guardado en: /content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a_fixed/gemma3_4b_4bit_fixed.jsonl

Procesando: gemma3_4b_bf16.jsonl
Cambios: 8

IDX 20
OLD: ['Arya I']
NEW: ['Jon II']

IDX 21
OLD: ['Arya I']
NEW: ['Jon II']

IDX 43
OLD: ['Tyrion III']
NEW: ['Tyrion III, Jon III']

IDX 62
OLD: ['Catelyn VII']
NEW: ['Catelyn VII, Tyrion V']

IDX 63
OLD: ['Catelyn VII']
NEW: ['Catelyn VII, Tyrion V']

IDX 72
OLD: ['Daenerys IV']
NEW: ['Daenerys V']

IDX 73
OLD: ['Daenerys IV']
NEW: ['Daenerys V']

IDX 91
OLD: ['Bran VI']
NEW: ['Catelyn X']

Guar

,file,idx,question,old,new
0,gemma3_4b_4bit.jsonl,20,"¿Quién le regala a Arya su espada, Aguja?",[Arya I],[Jon II]
1,gemma3_4b_4bit.jsonl,21,¿Qué consejo básico le da Jon a Arya sobre cóm...,[Arya I],[Jon II]
2,gemma3_4b_4bit.jsonl,43,¿Quién es el Lord Comandante de la Guardia de ...,[Tyrion III],"[Tyrion III, Jon III]"
3,gemma3_4b_4bit.jsonl,62,¿Quién lucha como campeón de Tyrion en su juic...,[Catelyn VII],"[Catelyn VII, Tyrion V]"
4,gemma3_4b_4bit.jsonl,63,¿Quién lucha como campeón de Lysa Arryn en el ...,[Catelyn VII],"[Catelyn VII, Tyrion V]"
5,gemma3_4b_4bit.jsonl,72,¿Qué ritual debe superar Daenerys en Vaes Doth...,[Daenerys IV],[Daenerys V]
6,gemma3_4b_4bit.jsonl,73,¿Qué gran destino profetizan para el hijo de D...,[Daenerys IV],[Daenerys V]
7,gemma3_4b_4bit.jsonl,91,¿Qué hace Viento Gris cuando el Gran Jon desaf...,[Bran VI],[Catelyn X]
8,gemma3_4b_bf16.jsonl,20,"¿Quién le regala a Arya su espada, Aguja?",[Arya I],[Jon II]
9,gemma3_4b_bf16.jsonl,21,¿Qué consejo básico le da Jon a Arya sobre cóm...,[Arya I],[Jon II]



Resumen guardado en:
/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs/q&a_fixed/changes_summary.csv
